# 07 - Reference Reproduction And Fair Comparison SER

Notebook này tạo một benchmark công bằng để kiểm tra lại hai nguồn reference có code local và so với mô hình của project.

## Mục tiêu

1. Tái tạo có kiểm soát các kiến trúc từ hai nguồn local:
   - **Emonity**: 1D-CNN, 2D-CNN, CNN-BiLSTM và validation-weighted ensemble.
   - **CNN/LSTM/CLSTM 4-dataset repo**: big 1D-CNN, LSTM và CNN-LSTM/CLSTM style model.
2. Thêm mô hình của project:
   - **ours_06d_coattention_light**: temporal branch + spectrogram branch + statistical branch + co-attention/gated fusion.
3. Dùng cùng một input, cùng một label set, cùng một split:
   - `paper_random_80_20_all4`: random sample-level, test 20%, validation lấy từ train.
   - `strict_speaker_all4`: speaker-aware split, không để cùng speaker xuất hiện ở train/validation/test.
4. Xuất bảng kết quả chung:
   - accuracy
   - macro-F1
   - weighted-F1
   - UAR
   - per-dataset accuracy/macro-F1
   - confusion matrix

## Accuracy gốc của hai nguồn có code

| Reference local | Dataset | Split trong code gốc | Accuracy gốc |
|---|---|---|---:|
| Emonity | CREMA-D + RAVDESS + TESS + SAVEE | random 60/20/20 từ `train_test_split` | ensemble test **88.50%** |
| CNN/LSTM/CLSTM 4-dataset repo | CREMA-D + RAVDESS + TESS + SAVEE | random 80/20 bằng `train_test_split(... test_size=0.2)` | evaluate trực tiếp **95.69%**, loaded best weights **96.91%** |

Lưu ý: notebook này không dùng lại kết quả gốc, mà train lại trên split chung để so sánh công bằng hơn.


## 1. Setup

Notebook ưu tiên dùng output `ser_processed` hoặc `ser_processed_update`. Dataset input cần có:

- `metadata.csv`
- folder audio model-ready, thường là `audio_16k/`

Nếu chạy trên Kaggle, bạn chỉ cần upload/add dataset chứa output của 01/02.


In [ ]:
from pathlib import Path
import os, json, time, random, zipfile, warnings, math

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = int(os.getenv("SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("CUDA devices:", torch.cuda.device_count())

PROJECT_ROOT = Path("/kaggle/working/Speech_Project") if Path("/kaggle/working").exists() else Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "07_Reference_Reproduction_Comparison_outputs"
CACHE_DIR = OUTPUT_DIR / "cache"
REPORT_DIR = OUTPUT_DIR / "reports"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
PRED_DIR = OUTPUT_DIR / "predictions"
for d in [OUTPUT_DIR, CACHE_DIR, REPORT_DIR, FIGURE_DIR, MODEL_DIR, PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

COMMON_EMOTIONS = ["neutral", "happy", "sad", "angry", "fear", "disgust"]
TARGET_SR = int(os.getenv("TARGET_SR", "16000"))
TARGET_DURATION = float(os.getenv("TARGET_DURATION", "4.0"))
TARGET_LENGTH = int(TARGET_SR * TARGET_DURATION)

N_MFCC = int(os.getenv("N_MFCC", "40"))
N_MELS = int(os.getenv("N_MELS", "96"))
N_FFT = int(os.getenv("N_FFT", "1024"))
HOP_LENGTH = int(os.getenv("HOP_LENGTH", "512"))

QUICK_RUN = os.getenv("QUICK_RUN", "0") == "1"
QUICK_RUN_SAMPLES = int(os.getenv("QUICK_RUN_SAMPLES", "900"))
MAX_EPOCHS = int(os.getenv("MAX_EPOCHS", "35"))
PATIENCE = int(os.getenv("PATIENCE", "8"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "64"))
NUM_WORKERS = int(os.getenv("NUM_WORKERS", "0"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE", "0.001"))

RUN_MODELS = os.getenv(
    "RUN_MODELS",
    "emonity_1d_cnn,emonity_2d_cnn,emonity_cnn_bilstm,clstm_big_cnn,clstm_lstm,clstm_cnn_lstm,ours_06d_coattention_light"
).split(",")
RUN_MODELS = [m.strip() for m in RUN_MODELS if m.strip()]

RUN_PROTOCOLS = os.getenv("RUN_PROTOCOLS", "paper_random_80_20_all4,strict_speaker_all4").split(",")
RUN_PROTOCOLS = [p.strip() for p in RUN_PROTOCOLS if p.strip()]
RUN_SINGLE_DATASET_PROTOCOLS = os.getenv("RUN_SINGLE_DATASET_PROTOCOLS", "1") == "1"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("QUICK_RUN:", QUICK_RUN, "MAX_EPOCHS:", MAX_EPOCHS)
print("RUN_MODELS:", RUN_MODELS)
print("RUN_PROTOCOLS:", RUN_PROTOCOLS)
print("RUN_SINGLE_DATASET_PROTOCOLS:", RUN_SINGLE_DATASET_PROTOCOLS)

## 2. Load `ser_processed` Metadata

Ta dùng cùng metadata/audio cho tất cả model. Điều này khác một số repo gốc, vì repo gốc thường tự extract feature và tự split riêng. Ở đây mục tiêu là **fair comparison**, nên mọi model phải dùng cùng dữ liệu và cùng split.


In [ ]:
def find_processed_root():
    candidates = [
        Path("/kaggle/input/ser-processed/ser_processed"),
        Path("/kaggle/input/ser-processed-update/ser_processed_update"),
        Path("/kaggle/input") / "ser_processed",
        Path("/kaggle/input") / "ser_processed_update",
        PROJECT_ROOT / "ser_processed",
        PROJECT_ROOT / "ser_processed_update",
        Path.cwd() / "ser_processed",
        Path.cwd() / "ser_processed_update",
        Path.cwd() / "01&02_Data_and_DataProcessing" / "ser_processed",
        Path.cwd() / "01&02_Data_and_DataProcessing" / "ser_processed_update",
    ]
    for root in candidates:
        if (root / "metadata.csv").exists():
            return root
    for base in [Path("/kaggle/input"), Path.cwd(), PROJECT_ROOT]:
        if base.exists():
            for meta in base.rglob("metadata.csv"):
                if "ser_processed" in str(meta.parent).lower():
                    return meta.parent
    raise FileNotFoundError("Cannot find ser_processed/ser_processed_update with metadata.csv")

PROCESSED_ROOT = find_processed_root()
metadata = pd.read_csv(PROCESSED_ROOT / "metadata.csv")

audio_dir_candidates = [
    PROCESSED_ROOT / "audio_16k",
    PROCESSED_ROOT / "audio_16k_clean_full",
    PROCESSED_ROOT / "audio",
]
AUDIO_DIR = next((p for p in audio_dir_candidates if p.exists()), None)

def infer_audio_path(row):
    sample_id = str(row.get("sample_id", ""))
    if AUDIO_DIR is not None and sample_id:
        p = AUDIO_DIR / f"{sample_id}.wav"
        if p.exists():
            return str(p)
    for col in ["audio_path", "processed_path", "filepath", "path"]:
        if col in row and pd.notna(row[col]) and Path(str(row[col])).exists():
            return str(row[col])
    return ""

metadata["audio_path"] = metadata.apply(infer_audio_path, axis=1)

speaker_col = "speaker_global" if "speaker_global" in metadata.columns else ("speaker_id" if "speaker_id" in metadata.columns else None)
if speaker_col is None:
    metadata["speaker_global"] = metadata["dataset"].astype(str) + "_" + metadata.index.astype(str)
    speaker_col = "speaker_global"

metadata = metadata[
    metadata["emotion"].isin(COMMON_EMOTIONS) &
    metadata["audio_path"].astype(str).str.len().gt(0)
].copy().reset_index(drop=True)

if QUICK_RUN and len(metadata) > QUICK_RUN_SAMPLES:
    metadata = metadata.groupby("emotion", group_keys=False).apply(
        lambda x: x.sample(min(len(x), max(1, QUICK_RUN_SAMPLES // len(COMMON_EMOTIONS))), random_state=SEED)
    ).reset_index(drop=True)

metadata["label_id"] = metadata["emotion"].map({e: i for i, e in enumerate(COMMON_EMOTIONS)}).astype(int)
metadata["dataset"] = metadata["dataset"].astype(str)
metadata["speaker_for_split"] = metadata[speaker_col].astype(str)

print("PROCESSED_ROOT:", PROCESSED_ROOT)
print("AUDIO_DIR:", AUDIO_DIR)
print("metadata:", metadata.shape)
display(pd.crosstab(metadata["dataset"], metadata["emotion"]).reindex(columns=COMMON_EMOTIONS))
print("Speakers:", metadata["speaker_for_split"].nunique())

## 3. Shared Split Protocols

Notebook này có ba kiểu đánh giá:

### `paper_random_80_20_all4`

Gần với cách các repo/paper accuracy cao thường làm: chia ngẫu nhiên theo sample trên dữ liệu gộp 4 dataset. Test chiếm 20%. Validation được tách từ phần train để không dùng test trong quá trình chọn epoch.

### `strict_speaker_all4`

Chia theo speaker trên dữ liệu gộp. Một speaker chỉ nằm trong một split. Protocol này khó hơn random vì model không được gặp cùng giọng ở train và test.

### Single-dataset protocols

Nếu `RUN_SINGLE_DATASET_PROTOCOLS=1`, notebook thêm:

- `single_random_80_20_<dataset>`: train/validation/test chỉ trong một dataset, random sample-level 60/20/20.
- `single_strict_speaker_<dataset>`: train/validation/test chỉ trong một dataset, chia theo speaker nếu dataset có đủ speaker.

Lưu ý: TESS chỉ có 2 speaker nên không đủ để chia strict thành train/validation/test độc lập. Notebook sẽ skip strict cho dataset nào có ít hơn 3 speaker.

In [ ]:
def make_random_80_20_split(df, source_indices=None):
    if source_indices is None:
        source_indices = np.arange(len(df))
    source_indices = np.asarray(source_indices)
    sub = df.iloc[source_indices].reset_index(drop=False)
    local_idx = np.arange(len(sub))
    train_val_local, test_local = train_test_split(
        local_idx, test_size=0.20, random_state=SEED, stratify=sub["label_id"]
    )
    train_local, val_local = train_test_split(
        train_val_local,
        test_size=0.25,
        random_state=SEED,
        stratify=sub.iloc[train_val_local]["label_id"],
    )
    return {
        "train": sub.iloc[train_local]["index"].values,
        "validation": sub.iloc[val_local]["index"].values,
        "test": sub.iloc[test_local]["index"].values,
    }


def make_strict_speaker_split(df, source_indices=None):
    if source_indices is None:
        source_indices = np.arange(len(df))
    source_indices = np.asarray(source_indices)
    sub = df.iloc[source_indices].reset_index(drop=False)
    groups = sub["speaker_for_split"].values
    if sub["speaker_for_split"].nunique() < 3:
        raise ValueError("Need at least 3 speakers for train/validation/test strict split")
    local_idx = np.arange(len(sub))
    outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    train_val_local, test_local = next(outer.split(local_idx, groups=groups))
    train_val_sub = sub.iloc[train_val_local]
    if train_val_sub["speaker_for_split"].nunique() < 2:
        raise ValueError("Not enough train-val speakers after test split")
    inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
    train_rel, val_rel = next(inner.split(train_val_sub, groups=train_val_sub["speaker_for_split"].values))
    train_local = train_val_local[train_rel]
    val_local = train_val_local[val_rel]
    return {
        "train": sub.iloc[train_local]["index"].values,
        "validation": sub.iloc[val_local]["index"].values,
        "test": sub.iloc[test_local]["index"].values,
    }

protocols = {}
if "paper_random_80_20_all4" in RUN_PROTOCOLS:
    protocols["paper_random_80_20_all4"] = make_random_80_20_split(metadata)
if "strict_speaker_all4" in RUN_PROTOCOLS:
    protocols["strict_speaker_all4"] = make_strict_speaker_split(metadata)

if RUN_SINGLE_DATASET_PROTOCOLS:
    for dataset_name in sorted(metadata["dataset"].unique()):
        dataset_indices = metadata.index[metadata["dataset"].eq(dataset_name)].values
        safe_dataset_name = str(dataset_name).lower().replace("-", "_").replace(" ", "_")
        if len(dataset_indices) < 30:
            print(f"Skip {dataset_name}: too few samples")
            continue
        try:
            protocols[f"single_random_80_20_{safe_dataset_name}"] = make_random_80_20_split(metadata, dataset_indices)
        except Exception as exc:
            print(f"Skip single random for {dataset_name}:", exc)
        try:
            protocols[f"single_strict_speaker_{safe_dataset_name}"] = make_strict_speaker_split(metadata, dataset_indices)
        except Exception as exc:
            print(f"Skip single strict for {dataset_name}:", exc)

for name, split in protocols.items():
    print("=" * 90)
    print(name, {k: len(v) for k, v in split.items()})
    for k, v in split.items():
        print(k)
        display(pd.crosstab(metadata.iloc[v]["dataset"], metadata.iloc[v]["emotion"]).reindex(columns=COMMON_EMOTIONS))
    if "strict" in name:
        sets = {k: set(metadata.iloc[v]["speaker_for_split"]) for k, v in split.items()}
        print("speaker overlap train/val:", len(sets["train"] & sets["validation"]))
        print("speaker overlap train/test:", len(sets["train"] & sets["test"]))
        print("speaker overlap val/test:", len(sets["validation"] & sets["test"]))

## Reference Preprocessing vs This Notebook

Hai repo local đều có phần xử lý data riêng, nhưng notebook này cố ý dùng pipeline chung để so sánh công bằng.

| Repo | Data processing gốc | Điểm có thể làm accuracy cao hơn | Notebook này làm gì |
|---|---|---|---|
| Emonity | Tự parse 4 dataset, extract nhiều nhóm feature cho 1D/2D, split random 60/20/20 | Random sample split, validation-weighted ensemble, feature pipeline riêng | Reimplement các kiến trúc chính trên cùng feature cache của project |
| CNN/LSTM/CLSTM | Load raw audio 2.5s offset 0.6, extract ZCR + RMSE + MFCC flatten, tạo thêm noise/pitch/pitch+noise trước khi split | Augmentation xảy ra trước split có thể làm bản gốc và bản augment của cùng file rơi vào train/test khác nhau | Reimplement big Conv1D/LSTM/CNN-LSTM nhưng dùng feature cache chung và split sạch hơn |

Vì vậy `6comparison` không phải là copy 100% preprocessing gốc. Nó là **controlled reproduction**: giữ ý tưởng kiến trúc, nhưng dùng cùng preprocessing và cùng split để đánh giá công bằng hơn. Nếu muốn kiểm chứng leakage/protocol gốc, cần chạy notebook gốc của từng repo riêng.

## 4. Shared Feature Extraction

Tất cả model dùng cùng feature cache để tránh mỗi kiến trúc được hưởng preprocessing riêng.

Feature được tạo:

- `X_temporal`: chuỗi frame-level `[T, 132]`: MFCC, delta, delta-delta, RMS, ZCR, spectral centroid, bandwidth, rolloff, contrast.
- `X_spectral`: ảnh phổ `[3, n_mels, T]`: log-Mel, delta log-Mel, delta-delta log-Mel.
- `X_stats`: vector thống kê từ temporal + chroma/flatness.
- `X_flat`: vector flatten mô phỏng cách repo CNN/LSTM/CLSTM đưa feature một chiều vào Conv1D.


In [ ]:
def crop_or_pad(y, target_len=TARGET_LENGTH):
    if len(y) > target_len:
        start = max(0, (len(y) - target_len) // 2)
        return y[start:start + target_len]
    if len(y) < target_len:
        return np.pad(y, (0, target_len - len(y)))
    return y

def load_audio(path):
    y, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    y = y.astype(np.float32)
    y = y - np.mean(y) if len(y) else y
    peak = np.max(np.abs(y)) if len(y) else 0.0
    if peak > 1.0:
        y = y / peak
    return crop_or_pad(y)

def pad_frames(x, target_frames):
    if x.shape[1] > target_frames:
        return x[:, :target_frames]
    if x.shape[1] < target_frames:
        return np.pad(x, ((0, 0), (0, target_frames - x.shape[1])), mode="edge")
    return x

def stats_for_matrix(mat):
    return np.concatenate([
        np.mean(mat, axis=1),
        np.std(mat, axis=1),
        np.min(mat, axis=1),
        np.max(mat, axis=1),
    ])

def extract_one(path):
    y = load_audio(path)
    mfcc = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    d1 = librosa.feature.delta(mfcc)
    d2 = librosa.feature.delta(mfcc, order=2)
    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP_LENGTH)
    centroid = librosa.feature.spectral_centroid(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    contrast = librosa.feature.spectral_contrast(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    temporal = np.vstack([mfcc, d1, d2, rms, zcr, centroid, bandwidth, rolloff, contrast]).astype(np.float32)

    mel = librosa.feature.melspectrogram(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
    logmel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
    dm = librosa.feature.delta(logmel).astype(np.float32)
    d2m = librosa.feature.delta(logmel, order=2).astype(np.float32)
    spectral = np.stack([logmel, dm, d2m], axis=0)

    chroma = librosa.feature.chroma_stft(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    flatness = librosa.feature.spectral_flatness(y=y, n_fft=N_FFT, hop_length=HOP_LENGTH)
    stats = np.concatenate([stats_for_matrix(temporal), stats_for_matrix(chroma), stats_for_matrix(flatness)]).astype(np.float32)

    return temporal.T, spectral, stats

cache_path = CACHE_DIR / f"features_{len(metadata)}samples_{int(TARGET_DURATION*1000)}ms_{N_MFCC}mfcc_{N_MELS}mel.npz"
if cache_path.exists():
    feature_cache = np.load(cache_path, allow_pickle=True)
    X_temporal = feature_cache["X_temporal"]
    X_spectral = feature_cache["X_spectral"]
    X_stats = feature_cache["X_stats"]
    print("Loaded cache:", cache_path)
else:
    temporals, spectrals, stats = [], [], []
    frame_counts = []
    for path in tqdm(metadata["audio_path"], desc="Extract shared features"):
        t, s, st = extract_one(path)
        temporals.append(t)
        spectrals.append(s)
        stats.append(st)
        frame_counts.append(t.shape[0])
    target_frames = int(np.median(frame_counts))
    target_frames = max(target_frames, 16)
    X_temporal = np.stack([pad_frames(t.T, target_frames).T for t in temporals]).astype(np.float32)
    X_spectral = np.stack([np.stack([pad_frames(ch, target_frames) for ch in s], axis=0) for s in spectrals]).astype(np.float32)
    X_stats = np.stack(stats).astype(np.float32)
    np.savez_compressed(cache_path, X_temporal=X_temporal, X_spectral=X_spectral, X_stats=X_stats, target_frames=np.asarray([target_frames]))
    print("Saved cache:", cache_path)

X_flat = X_temporal.reshape(len(X_temporal), -1).astype(np.float32)
y = metadata["label_id"].values.astype(np.int64)

print("X_temporal:", X_temporal.shape)
print("X_spectral:", X_spectral.shape)
print("X_stats:", X_stats.shape)
print("X_flat:", X_flat.shape)

## 5. Dataset Wrapper And Scaling

Scaler chỉ fit trên train split rồi transform validation/test. Điều này tránh leakage từ test vào preprocessing.


In [ ]:
RUN_ORIGINAL_LIKE_PIPELINES = os.getenv("RUN_ORIGINAL_LIKE_PIPELINES", "0") == "1"
ORIGINAL_AUGMENT_COPIES = int(os.getenv("ORIGINAL_AUGMENT_COPIES", "1"))


def fixed_audio_load(path, duration, offset=0.0, sr=None):
    sr = sr or TARGET_SR
    y, loaded_sr = librosa.load(path, sr=sr, mono=True, duration=duration, offset=offset)
    target_len = int(sr * duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    y = y.astype(np.float32)
    y = y - np.mean(y) if len(y) else y
    peak = np.max(np.abs(y)) if len(y) else 0.0
    if peak > 1.0:
        y = y / peak
    return y, sr


def ref_apply_noise(audio, low=0.001, high=0.010):
    noise_level = np.random.uniform(low, high)
    return (audio + noise_level * np.random.randn(len(audio))).astype(np.float32)


def ref_apply_pitch(audio, sr, n_steps=None):
    if n_steps is None:
        n_steps = np.random.uniform(-1.5, 1.5)
    try:
        return librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps).astype(np.float32)
    except Exception:
        return audio


def ref_apply_speed(audio, speed_factor=None):
    if speed_factor is None:
        speed_factor = np.random.uniform(0.95, 1.05)
    try:
        new_length = max(1, int(len(audio) / speed_factor))
        idx = np.linspace(0, len(audio) - 1, new_length).astype(int)
        out = audio[idx]
    except Exception:
        out = audio
    if len(out) < len(audio):
        out = np.pad(out, (0, len(audio) - len(out)))
    return out[:len(audio)].astype(np.float32)


def emonity_like_augment(audio, sr, prob=0.5):
    out = audio.copy()
    if np.random.random() < prob:
        out = ref_apply_noise(out, 0.001, 0.010)
    if np.random.random() < prob:
        out = ref_apply_pitch(out, sr)
    if np.random.random() < prob:
        out = ref_apply_speed(out)
    return out.astype(np.float32)


def resize_time(mat, target_length=128):
    mat = np.asarray(mat, dtype=np.float32)
    if mat.ndim == 1:
        mat = mat[None, :]
    if mat.shape[1] == target_length:
        return mat
    old = np.linspace(0, 1, mat.shape[1]) if mat.shape[1] > 1 else np.array([0.0])
    new = np.linspace(0, 1, target_length)
    return np.stack([np.interp(new, old, row) for row in mat], axis=0).astype(np.float32)


def extract_emonity_like_features(audio, sr, target_length=128):
    n_fft = min(2048, max(256, len(audio) // 4))
    hop = min(512, max(128, len(audio) // 8))
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20, n_fft=n_fft, hop_length=hop)
    if mfcc.shape[1] >= 9:
        mfcc = np.concatenate([mfcc, librosa.feature.delta(mfcc, width=5), librosa.feature.delta(mfcc, order=2, width=5)], axis=0)
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=64, n_fft=n_fft, hop_length=hop)
    logmel = librosa.power_to_db(mel, ref=np.max)
    centroid = librosa.feature.spectral_centroid(y=audio, sr=sr, n_fft=n_fft, hop_length=hop)
    bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sr, n_fft=n_fft, hop_length=hop)
    rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr, n_fft=n_fft, hop_length=hop)
    zcr = librosa.feature.zero_crossing_rate(audio, hop_length=hop)
    rms = librosa.feature.rms(y=audio, hop_length=hop)
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr, n_fft=n_fft, hop_length=hop)
    spectral = np.concatenate([centroid, bandwidth, rolloff], axis=0)
    x1d = np.concatenate([
        resize_time(mfcc, target_length),
        resize_time(spectral, target_length),
        resize_time(chroma, target_length),
        resize_time(zcr, target_length),
        resize_time(rms, target_length),
    ], axis=0).astype(np.float32)
    x2d = resize_time(logmel, target_length).astype(np.float32)
    x2d = np.stack([x2d, x2d, x2d], axis=0)
    return x1d.T, x2d


def clstm_noise(data):
    noise_amp = 0.035 * np.random.uniform() * max(np.max(np.abs(data)), 1e-6)
    return (data + noise_amp * np.random.normal(size=data.shape[0])).astype(np.float32)


def extract_clstm_like_flat(audio, sr=22050, frame_length=2048, hop_length=512):
    zcr = librosa.feature.zero_crossing_rate(audio, frame_length=frame_length, hop_length=hop_length).squeeze()
    rms = librosa.feature.rms(y=audio, frame_length=frame_length, hop_length=hop_length).squeeze()
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20, n_fft=frame_length, hop_length=hop_length).T.reshape(-1)
    return np.hstack([np.ravel(zcr), np.ravel(rms), mfcc]).astype(np.float32)


def pad_1d(vec, target_len):
    vec = np.asarray(vec, dtype=np.float32)
    if len(vec) > target_len:
        return vec[:target_len]
    if len(vec) < target_len:
        return np.pad(vec, (0, target_len - len(vec)))
    return vec


def build_original_like_feature_cache():
    cache_path = CACHE_DIR / f"original_like_pipelines_{len(metadata)}samples_aug{ORIGINAL_AUGMENT_COPIES}.npz"
    if cache_path.exists():
        print("Loaded original-like cache:", cache_path)
        return np.load(cache_path, allow_pickle=True)

    em_x1d, em_x2d, em_y, em_src = [], [], [], []
    cl_x, cl_y, cl_src = [], [], []

    for row in tqdm(metadata.itertuples(index=False), total=len(metadata), desc="Original-like pipelines"):
        path = row.audio_path
        label = int(row.label_id)

        # Emonity-like: fixed 3s + original + optional augmented copies.
        y3, sr3 = fixed_audio_load(path, duration=3.0, offset=0.0, sr=TARGET_SR)
        variants = [("original", y3)]
        for c in range(ORIGINAL_AUGMENT_COPIES):
            variants.append((f"augment_{c+1}", emonity_like_augment(y3, sr3)))
        for tag, audio in variants:
            x1d, x2d = extract_emonity_like_features(audio, sr3, target_length=128)
            em_x1d.append(x1d)
            em_x2d.append(x2d)
            em_y.append(label)
            em_src.append(f"{row.sample_id}:{tag}" if hasattr(row, "sample_id") else tag)

        # CNN/LSTM/CLSTM-like: duration 2.5s offset 0.6 + original/noise/pitch/pitch+noise.
        y25, sr25 = fixed_audio_load(path, duration=2.5, offset=0.6, sr=22050)
        cl_variants = [
            ("original", y25),
            ("noise", clstm_noise(y25)),
            ("pitch", ref_apply_pitch(y25, sr25, n_steps=0.7)),
            ("pitch_noise", clstm_noise(ref_apply_pitch(y25, sr25, n_steps=0.7))),
        ]
        for tag, audio in cl_variants:
            cl_x.append(extract_clstm_like_flat(audio, sr25))
            cl_y.append(label)
            cl_src.append(f"{row.sample_id}:{tag}" if hasattr(row, "sample_id") else tag)

    target_flat = max(len(v) for v in cl_x)
    cl_x = np.stack([pad_1d(v, target_flat) for v in cl_x]).astype(np.float32)
    em_x1d = np.stack(em_x1d).astype(np.float32)
    em_x2d = np.stack(em_x2d).astype(np.float32)
    em_y = np.asarray(em_y, dtype=np.int64)
    cl_y = np.asarray(cl_y, dtype=np.int64)
    np.savez_compressed(
        cache_path,
        emonity_x1d=em_x1d,
        emonity_x2d=em_x2d,
        emonity_y=em_y,
        emonity_source=np.asarray(em_src),
        clstm_flat=cl_x,
        clstm_y=cl_y,
        clstm_source=np.asarray(cl_src),
    )
    print("Saved original-like cache:", cache_path)
    print("Emonity-like:", em_x1d.shape, em_x2d.shape, em_y.shape)
    print("CLSTM-like:", cl_x.shape, cl_y.shape)
    return np.load(cache_path, allow_pickle=True)

if RUN_ORIGINAL_LIKE_PIPELINES:
    original_like_cache = build_original_like_feature_cache()
    print(original_like_cache.files)
else:
    print("RUN_ORIGINAL_LIKE_PIPELINES=0, skip original-like raw-audio feature pipelines.")

In [ ]:
class SerTensorDataset(Dataset):
    def __init__(self, indices, arrays, labels):
        self.indices = np.asarray(indices)
        self.arrays = arrays
        self.labels = labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        return {
            "temporal": torch.tensor(self.arrays["temporal"][idx], dtype=torch.float32),
            "spectral": torch.tensor(self.arrays["spectral"][idx], dtype=torch.float32),
            "stats": torch.tensor(self.arrays["stats"][idx], dtype=torch.float32),
            "flat": torch.tensor(self.arrays["flat"][idx], dtype=torch.float32),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
            "index": torch.tensor(idx, dtype=torch.long),
        }

def scale_for_split(split):
    train_idx = split["train"]
    arrays = {}

    flat_scaler = StandardScaler().fit(X_flat[train_idx])
    arrays["flat"] = flat_scaler.transform(X_flat).astype(np.float32)

    stats_scaler = StandardScaler().fit(X_stats[train_idx])
    arrays["stats"] = stats_scaler.transform(X_stats).astype(np.float32)

    temp_scaler = StandardScaler().fit(X_temporal[train_idx].reshape(-1, X_temporal.shape[-1]))
    arrays["temporal"] = temp_scaler.transform(X_temporal.reshape(-1, X_temporal.shape[-1])).reshape(X_temporal.shape).astype(np.float32)

    mean = X_spectral[train_idx].mean(axis=(0, 2, 3), keepdims=True)
    std = X_spectral[train_idx].std(axis=(0, 2, 3), keepdims=True) + 1e-6
    arrays["spectral"] = ((X_spectral - mean) / std).astype(np.float32)
    return arrays

def make_loaders(split, arrays):
    loaders = {}
    for split_name in ["train", "validation", "test"]:
        ds = SerTensorDataset(split[split_name], arrays, y)
        loaders[split_name] = DataLoader(
            ds,
            batch_size=BATCH_SIZE,
            shuffle=(split_name == "train"),
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
        )
    return loaders

## 6. Model Architectures

### Emonity-style

- `emonity_1d_cnn`: Conv1D trên chuỗi acoustic temporal.
- `emonity_2d_cnn`: Conv2D trên log-Mel spectrogram.
- `emonity_cnn_bilstm`: Conv1D trích pattern ngắn, BiLSTM học quan hệ dài hơn.

### CNN/LSTM/CLSTM 4-dataset-style

- `clstm_big_cnn`: big 1D-CNN trên vector feature flatten, gần với model đạt accuracy cao trong notebook gốc.
- `clstm_lstm`: LSTM trực tiếp trên chuỗi feature.
- `clstm_cnn_lstm`: Conv1D trước, LSTM sau.

### Our project model

- `ours_06d_coattention_light`: temporal + spectrogram + stats branch, sau đó fusion bằng gate/co-attention nhẹ.


In [ ]:
class Emonity1DCNN(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_dim, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.25),
            nn.Conv1d(128, 192, 5, padding=2), nn.BatchNorm1d(192), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.30),
            nn.Conv1d(192, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.35), nn.Linear(128, num_classes))

    def forward(self, batch):
        x = batch["temporal"].transpose(1, 2)
        return self.head(self.net(x))

class Emonity2DCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.15),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.20),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128, 128), nn.ReLU(), nn.Dropout(0.35), nn.Linear(128, num_classes))

    def forward(self, batch):
        return self.head(self.net(batch["spectral"]))

class EmonityCNNBiLSTM(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_dim, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.25),
            nn.Conv1d(128, 160, 3, padding=1), nn.BatchNorm1d(160), nn.ReLU(),
        )
        self.rnn = nn.LSTM(160, 96, batch_first=True, bidirectional=True)
        self.att = nn.Linear(192, 1)
        self.head = nn.Sequential(nn.Linear(192, 128), nn.ReLU(), nn.Dropout(0.35), nn.Linear(128, num_classes))

    def forward(self, batch):
        x = batch["temporal"].transpose(1, 2)
        x = self.conv(x).transpose(1, 2)
        h, _ = self.rnn(x)
        w = torch.softmax(self.att(h).squeeze(-1), dim=1).unsqueeze(-1)
        z = (h * w).sum(dim=1)
        return self.head(z)

class CLSTMBigCNN(nn.Module):
    def __init__(self, flat_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 512, 5, padding=2), nn.BatchNorm1d(512), nn.ReLU(), nn.MaxPool1d(5, stride=2, padding=2),
            nn.Conv1d(512, 512, 5, padding=2), nn.BatchNorm1d(512), nn.ReLU(), nn.MaxPool1d(5, stride=2, padding=2), nn.Dropout(0.20),
            nn.Conv1d(512, 256, 5, padding=2), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(5, stride=2, padding=2),
            nn.Conv1d(256, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(5, stride=2, padding=2), nn.Dropout(0.20),
            nn.Conv1d(256, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(32),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128 * 32, 512), nn.ReLU(), nn.LayerNorm(512), nn.Dropout(0.35), nn.Linear(512, num_classes))

    def forward(self, batch):
        return self.head(self.net(batch["flat"].unsqueeze(1)))

class CLSTMLSTM(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.rnn = nn.LSTM(in_dim, 128, batch_first=True, bidirectional=True, num_layers=1, dropout=0.0)
        self.head = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.35), nn.Linear(128, num_classes))

    def forward(self, batch):
        h, _ = self.rnn(batch["temporal"])
        return self.head(h[:, -1])

class CLSTMCNNLSTM(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_dim, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(128, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
        )
        self.rnn = nn.LSTM(128, 96, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Linear(192, 128), nn.ReLU(), nn.Dropout(0.35), nn.Linear(128, num_classes))

    def forward(self, batch):
        x = self.conv(batch["temporal"].transpose(1, 2)).transpose(1, 2)
        h, _ = self.rnn(x)
        return self.head(h.mean(dim=1))

class SEBlock2D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, max(4, channels // reduction)), nn.ReLU(),
            nn.Linear(max(4, channels // reduction), channels), nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = x.mean(dim=(2, 3))
        w = self.fc(w).view(b, c, 1, 1)
        return x * w

class Ours06DCoAttentionLight(nn.Module):
    def __init__(self, temporal_dim, stats_dim, num_classes):
        super().__init__()
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(temporal_dim, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.25),
            nn.Conv1d(128, 160, 3, padding=1), nn.BatchNorm1d(160), nn.ReLU(),
        )
        self.temporal_rnn = nn.LSTM(160, 96, batch_first=True, bidirectional=True)
        self.temporal_att = nn.Linear(192, 1)

        self.spectral_net = nn.Sequential(
            nn.Conv2d(3, 48, 3, padding=1), nn.BatchNorm2d(48), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(48, 96, 3, padding=1), nn.BatchNorm2d(96), nn.ReLU(), SEBlock2D(96), nn.MaxPool2d(2),
            nn.Conv2d(96, 160, 3, padding=1), nn.BatchNorm2d(160), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.stats_mlp = nn.Sequential(nn.Linear(stats_dim, 192), nn.LayerNorm(192), nn.GELU(), nn.Dropout(0.25), nn.Linear(192, 128), nn.GELU())

        self.t_proj = nn.Linear(192, 160)
        self.s_proj = nn.Linear(160, 160)
        self.gate = nn.Sequential(nn.Linear(160 * 2, 160), nn.Sigmoid())
        self.head = nn.Sequential(nn.Linear(160 + 128, 192), nn.LayerNorm(192), nn.GELU(), nn.Dropout(0.35), nn.Linear(192, num_classes))

    def forward(self, batch):
        t = self.temporal_conv(batch["temporal"].transpose(1, 2)).transpose(1, 2)
        h, _ = self.temporal_rnn(t)
        w = torch.softmax(self.temporal_att(h).squeeze(-1), dim=1).unsqueeze(-1)
        zt = (h * w).sum(dim=1)
        zt = self.t_proj(zt)

        zs = self.spectral_net(batch["spectral"]).flatten(1)
        zs = self.s_proj(zs)

        gate = self.gate(torch.cat([zt, zs], dim=1))
        z_acoustic = gate * zt + (1.0 - gate) * zs
        z_stats = self.stats_mlp(batch["stats"])
        return self.head(torch.cat([z_acoustic, z_stats], dim=1))

def build_model(model_name):
    temporal_dim = X_temporal.shape[-1]
    stats_dim = X_stats.shape[-1]
    flat_dim = X_flat.shape[-1]
    num_classes = len(COMMON_EMOTIONS)
    if model_name == "emonity_1d_cnn":
        return Emonity1DCNN(temporal_dim, num_classes)
    if model_name == "emonity_2d_cnn":
        return Emonity2DCNN(num_classes)
    if model_name == "emonity_cnn_bilstm":
        return EmonityCNNBiLSTM(temporal_dim, num_classes)
    if model_name == "clstm_big_cnn":
        return CLSTMBigCNN(flat_dim, num_classes)
    if model_name == "clstm_lstm":
        return CLSTMLSTM(temporal_dim, num_classes)
    if model_name == "clstm_cnn_lstm":
        return CLSTMCNNLSTM(temporal_dim, num_classes)
    if model_name == "ours_06d_coattention_light":
        return Ours06DCoAttentionLight(temporal_dim, stats_dim, num_classes)
    raise ValueError(model_name)

for m in RUN_MODELS:
    model = build_model(m)
    print(m, "parameters:", sum(p.numel() for p in model.parameters()))
    del model

## 7. Training And Evaluation

Metric chính:

- `accuracy`: giống đa số paper/repo đang báo cáo.
- `macro_f1`: công bằng hơn khi class không cân bằng.
- `uar`: unweighted average recall, thường dùng trong SER.


In [ ]:
def move_batch(batch):
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "uar": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }

@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    logits_all, y_all, idx_all = [], [], []
    for batch in loader:
        batch = move_batch(batch)
        logits = model(batch)
        logits_all.append(logits.detach().cpu())
        y_all.append(batch["label"].detach().cpu())
        idx_all.append(batch["index"].detach().cpu())
    return torch.cat(logits_all).numpy(), torch.cat(y_all).numpy(), torch.cat(idx_all).numpy()

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, n = 0.0, 0
    preds, labels = [], []
    for batch in loader:
        batch = move_batch(batch)
        logits = model(batch)
        loss = criterion(logits, batch["label"])
        if is_train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        total_loss += loss.item() * len(batch["label"])
        n += len(batch["label"])
        preds.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())
        labels.extend(batch["label"].detach().cpu().numpy().tolist())
    m = compute_metrics(labels, preds)
    m["loss"] = total_loss / max(n, 1)
    return m

def train_one_model(protocol_name, model_name, split, loaders):
    model = build_model(model_name).to(DEVICE)
    if torch.cuda.device_count() > 1 and os.getenv("USE_DATAPARALLEL", "0") == "1":
        model = nn.DataParallel(model)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

    best_score = -1.0
    best_state = None
    best_epoch = 0
    no_improve = 0
    history = []
    start = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        train_m = run_epoch(model, loaders["train"], criterion, optimizer)
        val_m = run_epoch(model, loaders["validation"], criterion)
        scheduler.step(val_m["macro_f1"])
        row = {"protocol": protocol_name, "model": model_name, "epoch": epoch, **{f"train_{k}": v for k, v in train_m.items()}, **{f"val_{k}": v for k, v in val_m.items()}}
        history.append(row)
        print(f"{protocol_name}/{model_name} epoch {epoch:02d} | train_f1={train_m['macro_f1']:.4f} | val_f1={val_m['macro_f1']:.4f}")
        if val_m["macro_f1"] > best_score:
            best_score = val_m["macro_f1"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    split_results = {}
    pred_paths = {}
    for split_name in ["train", "validation", "test"]:
        logits, yy, idxs = predict_logits(model, loaders[split_name])
        pp = torch.softmax(torch.tensor(logits), dim=1).numpy()
        pred = pp.argmax(axis=1)
        m = compute_metrics(yy, pred)
        m["loss"] = float(F.cross_entropy(torch.tensor(logits), torch.tensor(yy)).item())
        split_results[split_name] = m
        pred_df = pd.DataFrame({
            "row_index": idxs,
            "dataset": metadata.iloc[idxs]["dataset"].values,
            "speaker": metadata.iloc[idxs]["speaker_for_split"].values,
            "true": [COMMON_EMOTIONS[i] for i in yy],
            "pred": [COMMON_EMOTIONS[i] for i in pred],
        })
        for i, emotion in enumerate(COMMON_EMOTIONS):
            pred_df[f"p_{emotion}"] = pp[:, i]
        pred_path = PRED_DIR / f"predictions_{protocol_name}_{model_name}_{split_name}.csv"
        pred_df.to_csv(pred_path, index=False)
        pred_paths[split_name] = pred_path

    model_path = MODEL_DIR / f"{protocol_name}_{model_name}.pt"
    torch.save({"state_dict": model.state_dict(), "model_name": model_name, "protocol": protocol_name, "best_epoch": best_epoch, "best_val_macro_f1": best_score}, model_path)
    elapsed = time.time() - start
    return model, history, split_results, pred_paths, model_path, elapsed, best_epoch, best_score

def metrics_row(protocol, model, split_name, result, n, train_time_sec, best_epoch, best_val_macro_f1):
    row = {"protocol": protocol, "model": model, "split": split_name, "n": n, "train_time_sec": train_time_sec, "best_epoch": best_epoch, "best_val_macro_f1": best_val_macro_f1}
    row.update(result)
    return row

In [ ]:
all_metrics, all_history = [], []
prediction_registry = {}

for protocol_name, split in protocols.items():
    print("=" * 100)
    print("PROTOCOL:", protocol_name)
    arrays = scale_for_split(split)
    loaders = make_loaders(split, arrays)
    prediction_registry[protocol_name] = {}

    for model_name in RUN_MODELS:
        print("-" * 100)
        model, history, split_results, pred_paths, model_path, train_time, best_epoch, best_val = train_one_model(protocol_name, model_name, split, loaders)
        all_history.extend(history)
        prediction_registry[protocol_name][model_name] = pred_paths
        for split_name in ["train", "validation", "test"]:
            all_metrics.append(metrics_row(protocol_name, model_name, split_name, split_results[split_name], len(split[split_name]), train_time, best_epoch, best_val))
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

metrics_df = pd.DataFrame(all_metrics)
history_df = pd.DataFrame(all_history)
metrics_df.to_csv(REPORT_DIR / "07_reference_reproduction_metrics.csv", index=False)
history_df.to_csv(REPORT_DIR / "07_reference_reproduction_history.csv", index=False)

display(metrics_df[metrics_df["split"].eq("test")].sort_values(["protocol", "macro_f1"], ascending=[True, False]))
print("Saved:", REPORT_DIR / "07_reference_reproduction_metrics.csv")

## 8. Emonity-Style Validation-Weighted Ensemble

Emonity dùng ensemble dựa trên validation accuracy. Ở đây ta tái tạo ý tưởng đó bằng cách lấy prediction probability từ ba model:

- `emonity_1d_cnn`
- `emonity_2d_cnn`
- `emonity_cnn_bilstm`

Trọng số được tính từ validation macro-F1 để tránh dùng test chọn model.


In [ ]:
def load_pred(path):
    df = pd.read_csv(path)
    prob_cols = [f"p_{e}" for e in COMMON_EMOTIONS]
    probs = df[prob_cols].values.astype(np.float32)
    y_true = df["true"].map({e: i for i, e in enumerate(COMMON_EMOTIONS)}).values
    idx = df["row_index"].values
    return df, probs, y_true, idx

ensemble_rows = []
emonity_members = ["emonity_1d_cnn", "emonity_2d_cnn", "emonity_cnn_bilstm"]

for protocol_name in protocols:
    if not all(m in prediction_registry.get(protocol_name, {}) for m in emonity_members):
        continue
    val_scores = []
    for m in emonity_members:
        val_df, val_probs, val_y, _ = load_pred(prediction_registry[protocol_name][m]["validation"])
        val_pred = val_probs.argmax(axis=1)
        val_scores.append(max(1e-6, f1_score(val_y, val_pred, average="macro", zero_division=0)))
    weights = np.asarray(val_scores, dtype=np.float32)
    weights = weights / weights.sum()
    print(protocol_name, dict(zip(emonity_members, weights.round(4))))

    for split_name in ["validation", "test"]:
        base_df, probs0, yy, idx = load_pred(prediction_registry[protocol_name][emonity_members[0]][split_name])
        probs = np.zeros_like(probs0)
        for w, m in zip(weights, emonity_members):
            _, p, _, _ = load_pred(prediction_registry[protocol_name][m][split_name])
            probs += w * p
        pred = probs.argmax(axis=1)
        m = compute_metrics(yy, pred)
        row = metrics_row(protocol_name, "emonity_val_weighted_ensemble", split_name, m, len(yy), 0.0, 0, float(np.max(val_scores)))
        ensemble_rows.append(row)
        out_df = base_df[["row_index", "dataset", "speaker", "true"]].copy()
        out_df["pred"] = [COMMON_EMOTIONS[i] for i in pred]
        for i, emotion in enumerate(COMMON_EMOTIONS):
            out_df[f"p_{emotion}"] = probs[:, i]
        out_df.to_csv(PRED_DIR / f"predictions_{protocol_name}_emonity_val_weighted_ensemble_{split_name}.csv", index=False)

if ensemble_rows:
    ensemble_df = pd.DataFrame(ensemble_rows)
    metrics_df = pd.concat([metrics_df, ensemble_df], ignore_index=True)
    metrics_df.to_csv(REPORT_DIR / "07_reference_reproduction_metrics.csv", index=False)

display(metrics_df[metrics_df["split"].eq("test")].sort_values(["protocol", "macro_f1"], ascending=[True, False]))

## 9. Per-Dataset Metrics And Visualizations

Phần này rất quan trọng vì accuracy gộp có thể bị chi phối bởi CREMA-D hoặc TESS. Per-dataset metrics cho biết model mạnh/yếu trên từng corpus.


In [ ]:
per_dataset_rows = []
for pred_file in PRED_DIR.glob("predictions_*_test.csv"):
    name = pred_file.stem.replace("predictions_", "").replace("_test", "")
    parts = name.split("_")
    protocol = "paper_random_80_20_all4" if name.startswith("paper_random_80_20_all4") else "strict_speaker_all4"
    model = name.replace(protocol + "_", "")
    df = pd.read_csv(pred_file)
    for dataset, part in df.groupby("dataset"):
        yy = part["true"].map({e: i for i, e in enumerate(COMMON_EMOTIONS)}).values
        pp = part["pred"].map({e: i for i, e in enumerate(COMMON_EMOTIONS)}).values
        m = compute_metrics(yy, pp)
        per_dataset_rows.append({"protocol": protocol, "model": model, "dataset": dataset, "n": len(part), **m})

per_dataset_df = pd.DataFrame(per_dataset_rows)
per_dataset_df.to_csv(REPORT_DIR / "07_reference_reproduction_per_dataset_metrics.csv", index=False)
display(per_dataset_df.sort_values(["protocol", "model", "dataset"]))

test_df = metrics_df[metrics_df["split"].eq("test")].copy()
plt.figure(figsize=(14, 6))
sns.barplot(data=test_df, x="model", y="macro_f1", hue="protocol")
plt.xticks(rotation=35, ha="right")
plt.title("Test macro-F1 by model and protocol")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "07_test_macro_f1_by_model.png", dpi=180)
plt.show()

plt.figure(figsize=(14, 6))
sns.barplot(data=per_dataset_df, x="dataset", y="macro_f1", hue="model")
plt.title("Per-dataset test macro-F1")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "07_per_dataset_macro_f1.png", dpi=180)
plt.show()

## 10. Confusion Matrices For Best Models

Mỗi protocol chọn model tốt nhất theo test macro-F1 để vẽ confusion matrix. Khi viết báo cáo, có thể dùng hình này để phân tích emotion nào bị nhầm nhiều.


In [ ]:
for protocol_name in protocols:
    test_part = metrics_df[(metrics_df["protocol"].eq(protocol_name)) & (metrics_df["split"].eq("test"))].copy()
    if test_part.empty:
        continue
    best_model = test_part.sort_values("macro_f1", ascending=False).iloc[0]["model"]
    pred_path = PRED_DIR / f"predictions_{protocol_name}_{best_model}_test.csv"
    if not pred_path.exists():
        continue
    df = pd.read_csv(pred_path)
    yy = df["true"].map({e: i for i, e in enumerate(COMMON_EMOTIONS)}).values
    pp = df["pred"].map({e: i for i, e in enumerate(COMMON_EMOTIONS)}).values
    cm = confusion_matrix(yy, pp, labels=list(range(len(COMMON_EMOTIONS))))
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=COMMON_EMOTIONS, yticklabels=COMMON_EMOTIONS)
    plt.title(f"{protocol_name} - best model: {best_model}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    fig_path = FIGURE_DIR / f"confusion_{protocol_name}_{best_model}.png"
    plt.savefig(fig_path, dpi=180)
    plt.show()
    print("Saved:", fig_path)

## 11. Package Outputs

Zip này không chứa feature cache lớn mặc định. Nếu cần tải cả cache, đặt `PACKAGE_WITH_CACHE=1`.


In [ ]:
summary = {
    "notebook": "07_reference_reproduction_fair_comparison_SER",
    "objective": "Re-train local reference architectures and our 06D-style model under the same random and strict protocols.",
    "processed_root": str(PROCESSED_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "metrics_csv": str(REPORT_DIR / "07_reference_reproduction_metrics.csv"),
    "per_dataset_csv": str(REPORT_DIR / "07_reference_reproduction_per_dataset_metrics.csv"),
    "models": RUN_MODELS,
    "protocols": list(protocols.keys()),
    "run_single_dataset_protocols": RUN_SINGLE_DATASET_PROTOCOLS,
    "labels": COMMON_EMOTIONS,
    "reference_original_accuracy": {
        "Emonity ensemble test": 0.8850,
        "CNN_LSTM_CLSTM direct evaluate": 0.9569,
        "CNN_LSTM_CLSTM loaded best weights": 0.9691,
    },
}
with open(REPORT_DIR / "07_reference_reproduction_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

zip_path = PROJECT_ROOT / "07_Reference_Reproduction_Comparison_outputs_light.zip"
package_dirs = [REPORT_DIR, FIGURE_DIR, MODEL_DIR, PRED_DIR]
if os.getenv("PACKAGE_WITH_CACHE", "0") == "1":
    package_dirs.append(CACHE_DIR)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
    for folder in package_dirs:
        if folder.exists():
            for file_path in folder.rglob("*"):
                if file_path.is_file():
                    zf.write(file_path, file_path.relative_to(OUTPUT_DIR))

with zipfile.ZipFile(zip_path, "r") as zf:
    bad = zf.testzip()

print("ZIP:", zip_path)
print("ZIP integrity:", "OK" if bad is None else f"BAD: {bad}")
print("ZIP size MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))
zip_path